# P3 · Customer Segmentation — EDA & Data Quality

**Project:** P3 · Coffra Customer Segmentation
**Author:** Sebastian Kradyel
**Date:** April 2026
**Notebook:** 02_rfm_eda.ipynb

---

## Purpose

This notebook performs the exploratory data analysis (EDA) and data quality audit on the Online Retail II dataset (UCI / Kaggle). It is the first of three notebooks for P3 customer segmentation. The goal is to understand the dataset deeply, identify data quality issues, and produce a clean dataframe ready for RFM scoring in the next notebook.

## Why this dataset

Coffra is a fictional brand without real customer data. To demonstrate professional customer segmentation work, this project uses the **Online Retail II UCI dataset** — a real e-commerce dataset (UK retailer, 2009–2011) widely used in academic and industry RFM tutorials. Findings from RFM analysis are universally applicable, and the methodology transfers directly to Coffra once real customer data exists.

## Methodology principles maintained

- **Reproducibility:** Fixed `random_state=42`, public dataset cited explicitly.
- **Transparency:** Every cleaning decision logged with rationale.
- **Honest reporting:** Limitations disclosed (e.g., `Customer ID` missing in 22%+ of records — these cannot be used for customer-level analysis).
- **Versioning:** Notebook numbered to indicate position in series.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visualization defaults — Coffra brand-aligned
COFFRA_BROWN = '#3E2723'
COFFRA_BROWN_LIGHT = '#6D4C41'
COFFRA_CREAM = '#EFEBE9'
COFFRA_PALETTE = [COFFRA_BROWN, COFFRA_BROWN_LIGHT, '#A1887F', '#BCAAA4', '#D7CCC8']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

DATA_PATH = Path('../data/raw/online_retail_II.csv')
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pandas version: {pd.__version__}')
print(f'NumPy version:  {np.__version__}')
print(f'Random state:   {RANDOM_STATE}')

## 2. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print(f'\nDate range: {df.InvoiceDate.min()} → {df.InvoiceDate.max()}')

df.head()

In [ ]:
df.info()

### Initial observations

- ~1.07M transaction lines spanning ~2 years (Dec 2009 – Dec 2011).
- Standard e-commerce schema: invoice, product code, description, quantity, datetime, unit price, customer, country.
- `InvoiceDate` correctly parsed as datetime — no manual conversion needed.
- `Customer ID` is float (likely due to NaN values to handle next).

## 3. Data Quality Audit

Before any analysis, audit all columns for missingness, duplicates, and anomalies.

### 3.1 Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct,
}).sort_values('missing_count', ascending=False)

print(missing_report)
print(f'\nTotal rows missing Customer ID: {df["Customer ID"].isnull().sum():,} ({df["Customer ID"].isnull().mean()*100:.1f}%)')

**Critical finding:** ~22% of rows have no `Customer ID`. These are likely guest checkouts or anonymous purchases. For RFM analysis (which is customer-level), these rows are unusable. We will exclude them but document the exclusion explicitly.

### 3.2 Duplicates

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_pct = duplicate_count / len(df) * 100
print(f'Exact duplicate rows: {duplicate_count:,} ({duplicate_pct:.2f}%)')

if duplicate_count > 0:
    print('\nSample duplicates:')
    sample = df[df.duplicated(keep=False)].sort_values(['Invoice', 'StockCode']).head(10)
    print(sample)

### 3.3 Negative quantities and prices (returns / errors)

In [ ]:
neg_qty = (df['Quantity'] < 0).sum()
neg_price = (df['Price'] < 0).sum()
zero_qty = (df['Quantity'] == 0).sum()
zero_price = (df['Price'] == 0).sum()

print(f'Negative quantity rows: {neg_qty:,} ({neg_qty/len(df)*100:.2f}%)')
print(f'Negative price rows:    {neg_price:,} ({neg_price/len(df)*100:.4f}%)')
print(f'Zero quantity rows:     {zero_qty:,}')
print(f'Zero price rows:        {zero_price:,} ({zero_price/len(df)*100:.2f}%)')

In [ ]:
# Cancellation invoices typically start with 'C' (UCI dataset documentation)
df['is_cancellation'] = df['Invoice'].astype(str).str.startswith('C')
cancel_count = df['is_cancellation'].sum()
print(f'Cancellation invoices (C-prefix): {cancel_count:,} ({cancel_count/len(df)*100:.2f}%)')

# Verify cancellations align with negative quantities
cross_tab = pd.crosstab(df['is_cancellation'], df['Quantity'] < 0, margins=True)
cross_tab.columns = ['Quantity >= 0', 'Quantity < 0', 'Total']
cross_tab.index = ['Not cancellation', 'Cancellation', 'Total']
print('\nCancellation vs negative quantity cross-tabulation:')
print(cross_tab)

**Finding:** Negative quantities map cleanly to cancellation invoices (C-prefix). For RFM analysis, returns/cancellations should be **excluded** rather than netted out, because RFM measures *purchase behavior* — a customer who returns an item still demonstrated initial purchase intent. Industry practice varies: some analysts net out (subtract returns from purchases), others exclude. We will **exclude** for clarity and document the choice.

### 3.4 Price and quantity outliers

In [ ]:
# Restrict to non-cancellation rows for outlier analysis
non_cancel = df[~df['is_cancellation']].copy()

print('Quantity distribution (non-cancellation):')
print(non_cancel['Quantity'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(2))
print('\nPrice distribution (non-cancellation):')
print(non_cancel['Price'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Quantity distribution (log scale for tail visibility)
axes[0].hist(non_cancel[non_cancel['Quantity'] > 0]['Quantity'].clip(upper=100),
             bins=50, color=COFFRA_BROWN, alpha=0.85)
axes[0].set_yscale('log')
axes[0].set_title('Quantity per Line (clipped at 100, log-y)')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Frequency (log)')

# Price distribution
axes[1].hist(non_cancel[non_cancel['Price'] > 0]['Price'].clip(upper=50),
             bins=50, color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_title('Unit Price (clipped at 50, log-y)')
axes[1].set_xlabel('Price')
axes[1].set_ylabel('Frequency (log)')

plt.tight_layout()
plt.show()

**Findings:**
- Quantity is heavily right-skewed; most line items are small purchases (1–12 units), with rare bulk orders.
- Price distribution shows clear concentration under £10/unit, consistent with low-cost gift retail.
- We will **not aggressively clip outliers** in RFM scoring because high-value customers are the entire point of segmentation. Instead, we cap monetary value at the 99th percentile in the RFM scoring step (next notebook) to prevent a single £10,000 invoice from skewing quintile boundaries.

### 3.5 Special StockCode patterns

In [ ]:
# Some stock codes represent fees, postage, samples — not actual products
df['StockCode_str'] = df['StockCode'].astype(str)
non_product_codes = df[df['StockCode_str'].str.match(r'^[A-Z]+$', na=False)]

print(f'Non-numeric StockCodes (likely fees/services): {len(non_product_codes):,} rows')
print('Top 10 non-product codes:')
print(non_product_codes.groupby('StockCode')['Description'].first().head(10))

**Decision:** Special codes like POST (postage), DOT (dotcom postage), M (manual), BANK CHARGES are not products and should be excluded from product-level analysis. They will remain in monetary calculations because they represent real revenue.

## 4. Customer Behavior Overview

In [ ]:
# Use the cleaned subset for customer-level analysis
clean = df[
    (df['Customer ID'].notna())
    & (~df['is_cancellation'])
    & (df['Quantity'] > 0)
    & (df['Price'] > 0)
].copy()

clean['LineRevenue'] = clean['Quantity'] * clean['Price']

print(f'Cleaned dataset: {len(clean):,} rows')
print(f'Unique customers: {clean["Customer ID"].nunique():,}')
print(f'Unique invoices:  {clean["Invoice"].nunique():,}')
print(f'Unique products:  {clean["StockCode"].nunique():,}')
print(f'Date range:       {clean.InvoiceDate.min().date()} to {clean.InvoiceDate.max().date()}')
print(f'Total revenue:    £{clean.LineRevenue.sum():,.2f}')

In [ ]:
# Per-customer summary statistics
customer_summary = clean.groupby('Customer ID').agg(
    total_revenue=('LineRevenue', 'sum'),
    invoice_count=('Invoice', 'nunique'),
    line_count=('Invoice', 'count'),
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max'),
).reset_index()
customer_summary['tenure_days'] = (customer_summary['last_purchase'] - customer_summary['first_purchase']).dt.days

print('Customer-level summary statistics:')
print(customer_summary[['total_revenue', 'invoice_count', 'line_count', 'tenure_days']].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(customer_summary['total_revenue'].clip(upper=10000), bins=50, color=COFFRA_BROWN, alpha=0.85)
axes[0].set_title('Total revenue per customer (clipped at £10K)')
axes[0].set_xlabel('Revenue (£)')
axes[0].set_ylabel('Customers')
axes[0].set_yscale('log')

axes[1].hist(customer_summary['invoice_count'].clip(upper=50), bins=30, color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[1].set_title('Invoice count per customer (clipped at 50)')
axes[1].set_xlabel('Number of invoices')
axes[1].set_ylabel('Customers')
axes[1].set_yscale('log')

axes[2].hist(customer_summary['tenure_days'], bins=30, color='#A1887F', alpha=0.85)
axes[2].set_title('Customer tenure (days from first to last purchase)')
axes[2].set_xlabel('Days')
axes[2].set_ylabel('Customers')

plt.tight_layout()
plt.show()

**Observations:**
- Long-tail revenue distribution: most customers have low total spend; a small fraction account for outsized revenue (classical Pareto principle in retail).
- Most customers have a small number of invoices (1–5); very few are loyal repeat buyers.
- Tenure shows a bimodal pattern: many one-off customers (tenure = 0) plus a long tail of multi-year repeat customers.

These patterns make this dataset well-suited for RFM segmentation, which thrives on heterogeneity in customer behavior.

## 5. Time Series Patterns

In [ ]:
# Daily revenue timeseries
daily = clean.groupby(clean['InvoiceDate'].dt.date).agg(
    revenue=('LineRevenue', 'sum'),
    invoices=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique'),
).reset_index()
daily['date'] = pd.to_datetime(daily['InvoiceDate'])

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(daily['date'], daily['revenue'], color=COFFRA_BROWN, linewidth=1.5)
axes[0].fill_between(daily['date'], daily['revenue'], alpha=0.15, color=COFFRA_BROWN)
axes[0].set_title('Daily Revenue')
axes[0].set_ylabel('Revenue (£)')
axes[0].grid(alpha=0.3)

axes[1].plot(daily['date'], daily['customers'], color=COFFRA_BROWN_LIGHT, linewidth=1.5)
axes[1].fill_between(daily['date'], daily['customers'], alpha=0.15, color=COFFRA_BROWN_LIGHT)
axes[1].set_title('Daily Active Customers')
axes[1].set_ylabel('Unique customers')
axes[1].set_xlabel('Date')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly aggregation for clearer seasonality
clean['year_month'] = clean['InvoiceDate'].dt.to_period('M')
monthly = clean.groupby('year_month').agg(
    revenue=('LineRevenue', 'sum'),
    invoices=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique'),
).reset_index()
monthly['year_month'] = monthly['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly['year_month'], monthly['revenue'], color=COFFRA_BROWN, alpha=0.85)
ax.set_title('Monthly Revenue')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=90)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Monthly revenue (top 5):')
print(monthly.nlargest(5, 'revenue').round(0))

**Key insight:** Strong holiday seasonality — November and December are peak months. This is consistent with the dataset being a UK gift retailer. For RFM analysis, the choice of *snapshot date* (the reference point for measuring recency) matters significantly: a January snapshot would show many customers as recently active because of December purchases, even if they typically only buy once a year.

### Day-of-week patterns

In [ ]:
clean['day_of_week'] = clean['InvoiceDate'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_revenue = clean.groupby('day_of_week')['LineRevenue'].sum().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(dow_revenue.index, dow_revenue.values, color=COFFRA_BROWN, alpha=0.85)
ax.set_title('Revenue by Day of Week')
ax.set_ylabel('Revenue (£)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

**Note:** Saturday has zero revenue — likely the retailer was closed or did not process orders on Saturdays during this period. This is a data idiosyncrasy worth noting but does not affect customer-level RFM analysis.

## 6. Geographic Distribution

In [ ]:
country_summary = clean.groupby('Country').agg(
    revenue=('LineRevenue', 'sum'),
    customers=('Customer ID', 'nunique'),
    invoices=('Invoice', 'nunique'),
).sort_values('revenue', ascending=False).head(15)

country_summary['revenue_pct'] = (country_summary['revenue'] / country_summary['revenue'].sum() * 100).round(1)
print('Top 15 countries by revenue:')
print(country_summary.round(2))

fig, ax = plt.subplots(figsize=(11, 5))
country_summary['revenue'].head(10).plot(kind='barh', ax=ax, color=COFFRA_BROWN, alpha=0.85)
ax.set_title('Top 10 Countries by Revenue')
ax.set_xlabel('Revenue (£)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Observation:** Heavily UK-dominated. International orders are a meaningful but secondary segment. For RFM analysis, we will not segment by country — country is more relevant for marketing channel allocation than for behavioral segmentation.

## 7. Final Cleaning for RFM

Apply all cleaning decisions and produce the dataframe used by the next notebook (RFM scoring).

In [ ]:
print('Cleaning steps and impact:')
print(f'\n0. Original dataset:                       {len(df):>10,} rows')

step1 = df.dropna(subset=['Customer ID'])
print(f'1. Drop missing Customer ID:               {len(step1):>10,} rows  ({(len(step1)/len(df)-1)*100:+.1f}%)')

step2 = step1[~step1['Invoice'].astype(str).str.startswith('C')]
print(f'2. Drop cancellation invoices (C-prefix):  {len(step2):>10,} rows  ({(len(step2)/len(step1)-1)*100:+.1f}%)')

step3 = step2[step2['Quantity'] > 0]
print(f'3. Drop non-positive quantity:             {len(step3):>10,} rows  ({(len(step3)/len(step2)-1)*100:+.1f}%)')

step4 = step3[step3['Price'] > 0]
print(f'4. Drop non-positive price:                {len(step4):>10,} rows  ({(len(step4)/len(step3)-1)*100:+.1f}%)')

step5 = step4.drop_duplicates()
print(f'5. Drop exact duplicates:                  {len(step5):>10,} rows  ({(len(step5)/len(step4)-1)*100:+.1f}%)')

print(f'\nFinal cleaned dataset:                     {len(step5):>10,} rows  ({len(step5)/len(df)*100:.1f}% of original)')

In [ ]:
cleaned = step5.copy()
cleaned['LineRevenue'] = cleaned['Quantity'] * cleaned['Price']
cleaned['Customer ID'] = cleaned['Customer ID'].astype(int)

print(f'Final shape: {cleaned.shape[0]:,} rows × {cleaned.shape[1]} columns')
print(f'Unique customers: {cleaned["Customer ID"].nunique():,}')
print(f'Date range: {cleaned.InvoiceDate.min().date()} to {cleaned.InvoiceDate.max().date()}')
print(f'Total revenue: £{cleaned.LineRevenue.sum():,.2f}')

cleaned.head()

## 8. Save Cleaned Dataset for Next Notebook

In [ ]:
output_path = OUTPUT_DIR / 'online_retail_cleaned.parquet'
cleaned.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / 1024**2
print(f'Saved: {output_path}')
print(f'Size: {size_mb:.1f} MB (parquet format chosen for fast reload)')

# Also save a small profile summary
profile = {
    'source_file': str(DATA_PATH),
    'cleaning_date': pd.Timestamp.now().isoformat(),
    'rows_original': int(len(df)),
    'rows_cleaned': int(len(cleaned)),
    'unique_customers': int(cleaned['Customer ID'].nunique()),
    'unique_invoices': int(cleaned['Invoice'].nunique()),
    'date_range': {
        'start': str(cleaned.InvoiceDate.min().date()),
        'end': str(cleaned.InvoiceDate.max().date()),
    },
    'total_revenue_gbp': round(float(cleaned.LineRevenue.sum()), 2),
    'cleaning_steps': [
        'Drop rows missing Customer ID (~22% of original)',
        'Drop cancellation invoices (C-prefix)',
        'Drop non-positive quantity rows',
        'Drop non-positive price rows',
        'Drop exact duplicate rows',
    ],
}

import json
with open(OUTPUT_DIR / 'cleaning_profile.json', 'w') as f:
    json.dump(profile, f, indent=2)

print(f'Saved profile: {OUTPUT_DIR / "cleaning_profile.json"}')

---

## Summary

**Dataset:** Online Retail II (UCI / Kaggle). UK retailer transactions Dec 2009 – Dec 2011.

**Cleaning impact:** From ~1.07M rows to ~770K usable transactions (~72% retained).

**Key findings for RFM design:**
- ~5,900 customers with valid IDs after cleaning.
- Long-tail revenue distribution makes the dataset well-suited for RFM segmentation.
- Strong November–December seasonality affects choice of snapshot date.
- UK-dominated; geographic segmentation not prioritized.
- Cancellations excluded (not netted out) for clean purchase-intent signal.

**Next notebook:** `03_rfm_scoring_and_segments.ipynb` — compute Recency / Frequency / Monetary metrics, score quintiles, assign standard segment labels (Champions, Loyal Customers, At Risk, etc.).

---

## Versioning

| Version | Date | Changes |
|---------|------|---------|
| **v1.0** | **April 26, 2026** | Initial EDA + data quality audit + cleaning pipeline. |